# Chapter 8
Use the `cirq` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
import cirq
import numpy as np
import sympy

qubits = cirq.LineQubit.range(2)
x0, x1, theta = sympy.symbols("x0 x1 theta")

base_circuit = cirq.Circuit()
base_circuit.append(cirq.rx(x0)(qubits[0]))
base_circuit.append(cirq.ry(x1)(qubits[1]))
base_circuit.append(cirq.CNOT(*qubits))
base_circuit.append(cirq.ry(theta)(qubits[1]))
measured_circuit = base_circuit + cirq.measure(*qubits, key="z")

resolver = {x0: 0.2, x1: -0.4, theta: 0.7}
sampler = cirq.Simulator(seed=123)
samples = sampler.run(
    measured_circuit, param_resolver=resolver, repetitions=1000
)
print(measured_circuit)
print("moments:", len(measured_circuit))
for index, moment in enumerate(measured_circuit):
    print(index, moment.operations)
print("bitstring counts:", samples.histogram(key="z"))


In [ ]:
def encoding_block(qubits, features, layer_scale=1.0):
    """Yield two feature rotations per qubit at the given scale."""
    for index, qubit in enumerate(qubits):
        angle_x = layer_scale * features[index % len(features)]
        angle_z = layer_scale * features[(index + 1) % len(features)]
        yield cirq.rx(angle_x)(qubit)
        yield cirq.rz(angle_z)(qubit)

def trainable_block(qubits, symbols):
    """Yield one trainable rotation per qubit and a CNOT chain."""
    for qubit, symbol in zip(qubits, symbols):
        yield cirq.ry(symbol)(qubit)
    for left, right in zip(qubits[:-1], qubits[1:]):
        yield cirq.CNOT(left, right)

def reuploading_circuit(qubits, feature_symbols, weight_symbols, layers):
    """Alternate encoding and training; require one weight per qubit/block."""
    if layers < 1 or len(weight_symbols) != layers * len(qubits):
        raise ValueError("Need a positive depth and one weight per qubit/block")
    circuit = cirq.Circuit()
    for layer in range(layers):
        circuit.append(encoding_block(
            qubits, feature_symbols, layer_scale=layer + 1
        ))
        start = layer * len(qubits)
        stop = start + len(qubits)
        circuit.append(trainable_block(qubits, weight_symbols[start:stop]))
    return circuit

reupload_weights = sympy.symbols("w0:4")
reupload_example = reuploading_circuit(
    qubits, (x0, x1), reupload_weights, layers=2
)
print(reupload_example)


In [ ]:
class ZZFeatureGate(cirq.Gate):
    """Implement exp(-i * angle * Z tensor Z), with symbolic angles."""

    def __init__(self, angle):
        self.angle = angle

    def _num_qubits_(self):
        return 2

    def _is_parameterized_(self):
        return cirq.is_parameterized(self.angle)

    def _parameter_names_(self):
        return cirq.parameter_names(self.angle)

    def _resolve_parameters_(self, resolver, recursive):
        return ZZFeatureGate(resolver.value_of(self.angle, recursive))

    def _decompose_(self, qubits):
        left, right = qubits
        yield cirq.CNOT(left, right)
        yield cirq.rz(2.0 * self.angle)(right)
        yield cirq.CNOT(left, right)

    def _circuit_diagram_info_(self, args):
        return (f"ZZf({self.angle})", "ZZf")

q0, q1 = cirq.LineQubit.range(2)
phi = sympy.Symbol("phi")
custom_operation = ZZFeatureGate(phi).on(q0, q1)
custom_circuit = cirq.Circuit(cirq.H.on_each(q0, q1), custom_operation)
decomposed_circuit = cirq.Circuit(
    cirq.H.on_each(q0, q1), cirq.decompose_once(custom_operation)
)
print("custom circuit:\n", custom_circuit)
print("decomposition:\n", decomposed_circuit)
assert cirq.parameter_names(custom_circuit) == {"phi"}
resolved_custom = cirq.resolve_parameters(custom_circuit, {phi: 0.3})
resolved_decomposition = cirq.resolve_parameters(
    decomposed_circuit, {phi: 0.3}
)
assert not cirq.is_parameterized(resolved_custom)
np.testing.assert_allclose(
    cirq.unitary(resolved_custom), cirq.unitary(resolved_decomposition),
    atol=1e-7,
)
custom_result = cirq.Simulator(dtype=np.complex128).simulate(
    resolved_custom, qubit_order=[q0, q1]
)
print("resolved state:", np.round(custom_result.final_state_vector, 4))


In [ ]:
sweep_angle = sympy.Symbol("sweep_angle")
sweep_qubit = cirq.LineQubit(0)
sweep_circuit = cirq.Circuit(
    cirq.rx(0.7)(sweep_qubit),
    cirq.ry(sweep_angle)(sweep_qubit),
    cirq.measure(sweep_qubit, key="z"),
)
sweep = cirq.Linspace(
    key=sweep_angle.name, start=-np.pi, stop=np.pi, length=25
)
sweep_simulator = cirq.Simulator(seed=7)
sweep_results = sweep_simulator.run_sweep(
    sweep_circuit, params=sweep, repetitions=500
)
expectations = []
for setting, result in zip(sweep, sweep_results):
    bits = result.measurements["z"][:, 0]
    estimate = 1.0 - 2.0 * bits.mean()
    expectations.append(estimate)
    print(f"angle={setting[sweep_angle.name]: .3f}, <Z>={estimate: .3f}")


In [ ]:
def z_expectation_from_samples(result, key="z"):
    """Estimate Z from one single-qubit measurement under the given key."""
    bits = result.measurements[key][:, 0]
    return 1.0 - 2.0 * bits.mean()

def noisy_z_expectation(circuit, qubit, resolver, p, shots=2000):
    """Add local depolarization after each moment, then measure once."""
    if circuit.has_measurements():
        raise ValueError("Pass an unmeasured circuit")
    resolved = cirq.resolve_parameters(circuit, resolver)
    noise_model = cirq.ConstantQubitNoiseModel(
        cirq.depolarize(p), prepend=False
    )
    noisy = resolved.with_noise(noise_model)
    measured = noisy + cirq.measure(qubit, key="z")
    simulator = cirq.DensityMatrixSimulator(seed=11)
    result = simulator.run(measured, repetitions=shots)
    estimate = z_expectation_from_samples(result)
    standard_error = np.sqrt(max(0.0, 1.0 - estimate**2) / shots)
    return estimate, standard_error

noise_grid = [0.0, 0.001, 0.005, 0.01, 0.02]
ideal_z = cirq.Simulator().simulate_expectation_values(
    base_circuit, observables=[cirq.Z(qubits[0])],
    param_resolver=resolver,
)[0].real
print(f"ideal exact <Z>: {ideal_z:.4f}")
curve = []
for p in noise_grid:
    estimate, error = noisy_z_expectation(
        base_circuit, qubits[0], resolver, p
    )
    curve.append((p, estimate, error))
    print(f"p={p:.3f}, <Z>={estimate:.4f}, standard error={error:.4f}")


In [ ]:
import cirq
import numpy as np
import sympy
from scipy.optimize import minimize
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=80, noise=0.1, random_state=19)
x_train_raw, x_valid_raw, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=19
)
scaler = StandardScaler().fit(x_train_raw)
x_train = np.clip(scaler.transform(x_train_raw), -np.pi, np.pi)
x_valid = np.clip(scaler.transform(x_valid_raw), -np.pi, np.pi)
qubits = cirq.LineQubit.range(2)
feature_symbols = sympy.symbols("x0 x1")
weight_symbols = sympy.symbols("theta0:4")

def make_model_circuit():
    """Return a two-feature, four-angle circuit with a Z readout on qubit 0."""
    x0, x1 = feature_symbols
    t0, t1, t2, t3 = weight_symbols
    return cirq.Circuit(
        cirq.rx(x0)(qubits[0]), cirq.rx(x1)(qubits[1]),
        cirq.ry(t0)(qubits[0]), cirq.ry(t1)(qubits[1]),
        cirq.CNOT(qubits[0], qubits[1]),
        cirq.rx(t2)(qubits[0]), cirq.rx(t3)(qubits[1]),
        cirq.CNOT(qubits[1], qubits[0]),
    )

model_circuit = make_model_circuit()
observable = cirq.Z(qubits[0])
simulator = cirq.Simulator(dtype=np.complex128, seed=19)

def score(features, weights):
    """Return the exact Z expectation for one feature row and angle vector."""
    bindings = dict(zip(feature_symbols, features))
    bindings.update(dict(zip(weight_symbols, weights)))
    state = simulator.simulate(
        model_circuit, param_resolver=bindings, qubit_order=qubits
    ).final_state_vector
    return float(observable.expectation_from_state_vector(
        state, qubit_map={qubit: index for index, qubit in enumerate(qubits)}
    ).real)

def probability(features, weights):
    """Assign class 1 to computational-basis outcome 1 on the readout qubit."""
    return 0.5 * (1.0 - score(features, weights))

def binary_cross_entropy(weights, features, labels):
    """Return mean binary cross-entropy with clipping for stable logs."""
    probabilities = np.array([probability(x, weights) for x in features])
    probabilities = np.clip(probabilities, 1e-6, 1.0 - 1e-6)
    return -np.mean(
        labels * np.log(probabilities)
        + (1 - labels) * np.log(1 - probabilities)
    )

initial_weights = np.random.default_rng(19).normal(
    scale=0.3, size=len(weight_symbols)
)
initial_loss = binary_cross_entropy(initial_weights, x_train, y_train)
fit_result = minimize(
    binary_cross_entropy, initial_weights, args=(x_train, y_train),
    method="Nelder-Mead", options={"maxiter": 350},
)
valid_probabilities = np.array([
    probability(x, fit_result.x) for x in x_valid
])
print("optimizer:", fit_result.success, fit_result.message)
print("training loss:", initial_loss, "->", fit_result.fun)
print("validation accuracy:", accuracy_score(
    y_valid, valid_probabilities >= 0.5
))
print("first five validation probabilities:", valid_probabilities[:5])


In [ ]:
def parameter_shift_gradient(features, weights, parameter_index):
    """Differentiate the expectation for a once-used Pauli rotation angle."""
    shift = np.zeros_like(weights)
    shift[parameter_index] = np.pi / 2.0
    plus = score(features, weights + shift)
    minus = score(features, weights - shift)
    return 0.5 * (plus - minus)

def gradient_vector(features, weights):
    return np.array([
        parameter_shift_gradient(features, weights, index)
        for index in range(len(weights))
    ])

probe_weights = initial_weights.copy()
shift_gradient = np.array([
    parameter_shift_gradient(x_valid[0], probe_weights, j)
    for j in range(len(probe_weights))
])
print("expectation gradient:", shift_gradient)
